# Sistema de Recomendação de Espetáculos Teatrais — Portal Aventura
## Modelagem: Filtragem Colaborativa Baseada em Itens com KNN

---

**Projeto:** Sistema de Recomendação — Portal Aventura Teatros  
**Etapa:** Modelagem — KNN Item-Based Collaborative Filtering  
**Algoritmo:** K-Nearest Neighbors com Similaridade por Cosseno  

---

### Contexto

O Portal Aventura ainda não possui um mecanismo estruturado de recomendação de espetáculos. Na prática, isso limita a descoberta de novas experiências: o usuário acessa um espetáculo específico, mas não recebe sugestões relacionadas ao seu comportamento ou a perfis semelhantes.

Este notebook implementa a etapa de modelagem utilizando o dataset balanceado preparado na etapa anterior. O objetivo é construir um modelo de filtragem colaborativa baseada em itens capaz de, dado um espetáculo de referência, identificar e recomendar outros espetáculos com padrões de avaliação similares entre os usuários.

Ao final do pipeline, são gerados três artefatos principais:

| Arquivo | Finalidade |
|---|---|
| `modelos/modelo_knn.pkl` | Modelo KNN treinado, pronto para inferência |
| `modelos/matriz_esparsa.pkl` | Matriz CSR utilizada no treinamento |
| `modelos/mapeamentos_itens.pkl` | Dicionários de índice para decodificação dos resultados |

---

### Estrutura do Notebook

| Secao | Descricao |
|---|---|
| 1. Configuracao do Ambiente | Imports e configuracoes globais |
| 2. Conexao com a Base da Sprint 1 | Analise critica do dataset gerado na etapa anterior |
| 3. Carregamento dos Dados | Leitura do dataset balanceado |
| 4. Analise Exploratoria | Visao geral do dataset de modelagem |
| 5. Construcao da Matriz Item-Usuario | Pivot table, esparsidade e matriz CSR |
| 6. Fundamentacao Teorica | Item-Based vs User-Based e Similaridade Cosseno |
| 7. Modelagem com KNN | Treinamento com NearestNeighbors |
| 8. Funcao de Recomendacao | Implementacao, inferencia e exemplos praticos |
| 9. Avaliacao do Modelo | RMSE e MAE |
| 10. Preparacao para Comparacao com SVD | Estrutura comparativa |
| 11. Exportacao do Modelo | Serializacao com joblib |
| 12. Conclusao | Descobertas, metricas e proximos passos |

---
## 1. Configuracao do Ambiente

Antes de iniciar a modelagem, importamos todas as bibliotecas necessarias. A organizacao dos imports por categoria facilita a manutencao e leitura do codigo.

In [ ]:
# Bibliotecas principais
import pandas as pd
import numpy as np

# Visualiza??o
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

# Modelo e avalia??o
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

# Serializa??o
import joblib
import os

# Estilo de visualiza??o
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

print('Ambiente configurado')
print(f'Pandas  : {pd.__version__}')
print(f'NumPy   : {np.__version__}')
print(f'Sklearn : {__import__("sklearn").__version__}')


---
## 2. Conexao com a Base da Sprint 1

Esta secao estabelece a conexao critica entre o pipeline de preparacao de dados da etapa anterior e o modelo construido aqui. O objetivo nao e reprocessar os dados, mas sim analisar como as decisoes metodologicas da Sprint 1 impactam diretamente a qualidade e as limitacoes deste modelo.

### 2.1 O que veio da Sprint 1

A base original continha registros transacionais de compras de ingressos, sem nenhuma avaliacao explicita dos usuarios. As principais transformacoes realizadas na Sprint 1 foram:

| Decisao | Descricao | Impacto neste modelo |
|---|---|---|
| Feedback implicito | Rating derivado de recorrencia (60%), valor do ingresso (25%) e antecedencia da compra (15%) | Os ratings que alimentam o KNN nao sao percepcoes subjetivas do usuario, mas proxies comportamentais |
| Remocao do outlier (ID 68564) | Usuario com 79.505 registros foi removido | Preveniu que um unico perfil distorcesse toda a matriz de similaridade |
| Filtro de cold-start | Usuarios com menos de 3 interacoes e itens com menos de 10 foram removidos | Garante que todos os vetores na matriz tenham densidade minima para calculo de similaridade |
| Balanceamento por quintis | Ratings distribuidos uniformemente de 1 a 5 por quantil do score continuo | A distribuicao balanceada evita que a similaridade cosseno seja dominada por uma unica faixa de rating |
| Codificacao de IDs | `id_user` e `nome_evento` convertidos em indices inteiros continuos | Compatibilidade direta com a matriz numerica exigida pelo NearestNeighbors |

### 2.2 Licoes aprendidas e impacto nos resultados

A substituicao de ratings sinteticos por feedback implicito foi a decisao mais relevante da Sprint 1 para a qualidade deste modelo. Ratings gerados artificialmente com distribuicao probabilistica introduziriam ruido sistematico na matriz de similaridade — dois espetaculos pareceriam similares nao porque usuarios com perfis parecidos os consumiram, mas por coincidencia da distribuicao aleatoria dos numeros.

Com o score implicito, a similaridade cosseno calculada nesta etapa reflete padroes reais de comportamento: espetaculos que atraem o mesmo perfil de usuario engajado (que volta, paga mais ou compra com antecedencia) tendem a ter vetores mais proximos no espaco matricial.

### 2.3 Limitacoes herdadas

Tres limitacoes relevantes da Sprint 1 se propagam para este modelo:

1. **Esparsidade elevada (98,62%)**: mesmo apos os filtros de cold-start, a grande maioria dos pares usuario-item nao possui interacao. Isso reduz a cobertura das predicoes do KNN — pares sem vizinhos avaliados em comum nao geram estimativa de rating.

2. **Ausencia de atributos de conteudo**: o modelo usa apenas ratings. Espetaculos da mesma categoria ou do mesmo tipo de producao poderiam ser penalizados por esparsidade, mesmo sendo objetivamente similares. A integracao dos metadados de `itens_metadata.csv` e um passo natural para expandir o modelo.

3. **Hipoteses de ponderacao nao validadas empiricamente**: os pesos 60/25/15 do score implicito foram definidos com criterio metodologico, mas nao testados contra uma metrica de satisfacao real do usuario. Ajustes nesses pesos podem alterar a ordenacao dos ratings e, consequentemente, os resultados do modelo.

---
## 3. Carregamento dos Dados

Nesta etapa, o dataset balanceado gerado na Sprint 1 e carregado a partir do arquivo versionado em `src/dataset/recomendacao_balanceado.csv`. Tambem sao carregados os metadados dos itens para permitir a interpretacao dos resultados.

As colunas obrigatorias para a modelagem sao:

| Coluna | Tipo | Descricao |
|---|---|---|
| `user_id` | int | Indice numerico do usuario (codificado na Sprint 1) |
| `item_id` | int | Indice numerico do espetaculo (codificado na Sprint 1) |
| `rating` | float | Rating de 1 a 5 derivado do score implicito por quintis |

In [ ]:
# Carregar dados preparados no tratamento
DATASET_PATH = '../dataset/recomendacao_balanceado.csv'
ITEM_META_PATH = '../dataset/itens_metadata.csv'

df_raw = pd.read_csv(DATASET_PATH)
item_meta = pd.read_csv(ITEM_META_PATH)

# Validar colunas m?nimas para a modelagem
colunas_necessarias = ['user_id', 'item_id', 'rating']
assert all(col in df_raw.columns for col in colunas_necessarias),     f'Dataset precisa conter: {colunas_necessarias}'

df = df_raw[colunas_necessarias].copy()

print('Datasets carregados')
print(f'Registros de modelagem : {len(df):,}')
print(f'Itens com metadados    : {len(item_meta):,}')
df.head(10)


---
## 4. Checagem Exploratória do Dataset de Modelagem

A análise exploratória completa foi realizada no notebook `tratamento-dados.ipynb`. Nesta etapa, fazemos apenas uma checagem direcionada à modelagem: volume de usuários e itens, distribuição dos ratings, densidade da matriz e quantidade de interações por usuário e espetáculo. Esses pontos são suficientes para validar se o dataset final está adequado ao KNN e para interpretar as limitações do modelo.

In [ ]:
# Checagem mínima necessária antes da modelagem
n_usuarios = df['user_id'].nunique()
n_itens    = df['item_id'].nunique()
n_ratings  = len(df)
densidade  = n_ratings / (n_usuarios * n_itens) * 100

print('=== Informacoes Gerais do Dataset de Modelagem ===')
print(f'   Usuarios unicos        : {n_usuarios:>10,}')
print(f'   Espetaculos unicos     : {n_itens:>10,}')
print(f'   Total de interacoes    : {n_ratings:>10,}')
print(f'   Densidade da matriz    : {densidade:>9.4f}%')
print(f'   Rating medio           : {df["rating"].mean():>10.2f}')
print(f'   Rating minimo          : {df["rating"].min():>10.0f}')
print(f'   Rating maximo          : {df["rating"].max():>10.0f}')

print('\nValores nulos por coluna:')
print(df.isnull().sum().to_string())

print('\nInteracoes por usuario:')
display(df.groupby('user_id').size().describe().rename('interacoes').to_frame())

print('\nInteracoes por espetaculo:')
display(df.groupby('item_id').size().describe().rename('interacoes').to_frame())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Analise Exploratoria — Dataset Balanceado (Portal Aventura)',
             fontsize=13, fontweight='bold')

colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60']

# Distribuicao dos ratings
rating_counts = df['rating'].value_counts().sort_index()
bars = axes[0].bar(rating_counts.index, rating_counts.values,
                   color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_title('Distribuicao dos Ratings', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Avaliacao')
axes[0].set_ylabel('Quantidade de Registros')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, val in zip(bars, rating_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{val:,}', ha='center', va='bottom', fontsize=9)

# Top 15 espetaculos mais avaliados
top_itens = df['item_id'].value_counts().head(15)
axes[1].barh(range(len(top_itens)), top_itens.values[::-1],
             color='coral', edgecolor='white')
axes[1].set_yticks(range(len(top_itens)))
axes[1].set_yticklabels([f'Item {i}' for i in top_itens.index[::-1]], fontsize=8)
axes[1].set_title('Top 15 Espetaculos por Interacoes', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Numero de Avaliacoes')

# Distribuicao de avaliacoes por usuario
avals_por_usuario = df.groupby('user_id')['rating'].count()
axes[2].hist(avals_por_usuario, bins=30, color='steelblue', edgecolor='white')
axes[2].set_title('Interacoes por Usuario', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Numero de Avaliacoes')
axes[2].set_ylabel('Numero de Usuarios')

plt.tight_layout()
plt.savefig('eda_modelagem.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 5. Construcao da Matriz Item-Usuario

### Por que precisamos de uma matriz?

O KNN opera em um espaco vetorial: cada espetaculo e representado como um vetor cujas dimensoes correspondem as avaliacoes dos usuarios. Para calcular a similaridade entre dois espetaculos, e preciso que ambos existam no mesmo espaco vetorial.

A Pivot Table transforma o formato longo `(user_id, item_id, rating)` em uma matriz estruturada:

| | user_0 | user_1 | user_2 | ... |
|---|---|---|---|---|
| **item_0** | 4.0 | NaN | 3.0 | ... |
| **item_1** | NaN | 5.0 | NaN | ... |
| **item_2** | 2.0 | 3.0 | NaN | ... |

- **Linhas** = Itens (espetaculos) — posicao no espaco vetorial
- **Colunas** = Usuarios — dimensoes do vetor
- **NaN** = Usuario nao avaliou o item (celula vazia — gera esparsidade)

Os itens ficam nas linhas porque realizamos filtragem baseada em itens (Item-Based CF). O modelo compara vetores de itens, nao de usuarios.

In [ ]:
# Itens ficam nas linhas porque o modelo é item-based.
matriz_item_usuario = df.pivot_table(
    index='item_id',
    columns='user_id',
    values='rating',
    aggfunc='mean'
)

print('=== Matriz Item-Usuario ===')
print(f'   Dimensoes  : {matriz_item_usuario.shape[0]:,} itens x {matriz_item_usuario.shape[1]:,} usuarios')

total_celulas       = matriz_item_usuario.size
celulas_preenchidas = matriz_item_usuario.notna().sum().sum()
esparsidade         = (1 - celulas_preenchidas / total_celulas) * 100

print(f'\n   Celulas preenchidas : {celulas_preenchidas:>10,}')
print(f'   Celulas totais      : {total_celulas:>10,}')
print(f'   Esparsidade         : {esparsidade:>9.4f}%')

print('\nAmostra da Matriz (5x5):')
display(matriz_item_usuario.iloc[:5, :5])

In [ ]:
# Zero representa ausência de interação na matriz de similaridade.
matriz_preenchida = matriz_item_usuario.fillna(0)

matriz_esparsa = csr_matrix(matriz_preenchida.values)

# Mapeamentos necessários para converter índices da matriz em item_ids.
item_ids    = list(matriz_item_usuario.index)
item_to_idx = {item: idx for idx, item in enumerate(item_ids)}   # item_id -> indice
idx_to_item = {idx: item for item, idx in item_to_idx.items()}   # indice -> item_id

print('=== Matriz Esparsa Criada ===')
print(f'   Tipo           : {type(matriz_esparsa).__name__}')
print(f'   Shape          : {matriz_esparsa.shape}')
print(f'   Itens mapeados : {len(item_ids):,}')

---
## 6. Fundamentacao Teorica

### 6.1 Por que Item-Based em vez de User-Based?

A filtragem colaborativa pode ser orientada a usuarios (encontrar usuarios similares e recomendar o que eles avaliaram positivamente) ou a itens (encontrar itens similares ao que o usuario ja avaliou). Para o contexto do Portal Aventura, a abordagem Item-Based e a mais adequada:

| Criterio | User-Based | Item-Based |
|---|---|---|
| Escalabilidade | Cresce com o numero de usuarios | Estavel — o catalogo de espetaculos e fixo por temporada |
| Estabilidade | Perfis de usuarios mudam com o tempo | Padroes de avaliacao por espetaculo sao estaveis apos a temporada |
| Esparsidade | Sensivel — poucos usuarios em comum entre pares | Mais robusto a matrizes com alta esparsidade |
| Interpretabilidade | Moderada | Alta — quem gostou de A tambem gostou de B |
| Novos usuarios | Dificil — cold start severo | Recomenda com base no item acessado, sem historico do usuario |

No Portal Aventura, o catalogo de espetaculos e fixo apos o encerramento de cada temporada, enquanto novos usuarios chegam continuamente. A abordagem Item-Based e, portanto, a escolha mais escalavel e estavel para esse contexto.

---

### 6.2 A Matematica da Similaridade por Cosseno

A similaridade por cosseno mede o angulo entre dois vetores no espaco multidimensional, sendo independente da magnitude (escala das avaliacoes):

$$\text{sim}(A, B) = \cos(\theta) = \frac{\vec{A} \cdot \vec{B}}{\|\vec{A}\| \cdot \|\vec{B}\|}$$

Onde:
- $\vec{A} \cdot \vec{B} = \sum_{u \in U} r_{u,A} \cdot r_{u,B}$ — produto escalar dos ratings
- $\|\vec{A}\| = \sqrt{\sum_{u \in U} r_{u,A}^2}$ — norma euclidiana do vetor A
- Resultado varia de **0** (sem similaridade) a **1** (vetores identicos)

**Exemplo numerico com espetaculos do Portal Aventura:**

| Usuario | Espetaculo A | Espetaculo B |
|---|---|---|
| user_1 | 5 | 4 |
| user_2 | 4 | 5 |
| user_3 | 0 | 0 |
| user_4 | 3 | 3 |

- Produto escalar: (5x4) + (4x5) + (0x0) + (3x3) = 20 + 20 + 0 + 9 = 49
- Norma A: raiz(25 + 16 + 0 + 9) = raiz(50) ≈ 7,07
- Norma B: raiz(16 + 25 + 0 + 9) = raiz(50) ≈ 7,07
- Similaridade: 49 / (7,07 x 7,07) ≈ **0,98** — espetaculos muito similares

O scikit-learn usa `metric='cosine'` que calcula a distancia cosseno = 1 - similaridade. Portanto, menor distancia equivale a maior similaridade.

---

### 6.3 O Algoritmo KNN Aplicado ao Portal Aventura

O processo de recomendacao segue os seguintes passos:

1. Um usuario acessa a pagina do espetaculo X no portal
2. O vetor do espetaculo X e recuperado da matriz Item-Usuario
3. O KNN calcula a distancia cosseno entre X e todos os outros espetaculos
4. Os `k` espetaculos com menor distancia (maior similaridade) sao retornados
5. O portal exibe esses espetaculos na secao de recomendacoes da pagina

---
## 7. Modelagem com KNN

O `NearestNeighbors` do scikit-learn implementa a busca pelos vizinhos mais proximos. Optamos por `algorithm='brute'` (busca exaustiva) por ser o unico algoritmo do scikit-learn compativel com a metrica cosseno em matrizes esparsas. Para o tamanho do catalogo do Portal Aventura, o custo computacional e aceitavel.

In [ ]:
# Hiperparâmetros do KNN item-based
N_VIZINHOS = 20        # numero de vizinhos a buscar (inclui o proprio item)
METRICA    = 'cosine'  # distancia cosseno = 1 - similaridade_cosseno
ALGORITMO  = 'brute'   # busca exaustiva: unico compativel com cosseno + CSR

modelo_knn = NearestNeighbors(
    n_neighbors=N_VIZINHOS,
    metric=METRICA,
    algorithm=ALGORITMO,
    n_jobs=-1
)

modelo_knn.fit(matriz_esparsa)

print('=== Modelo KNN Treinado ===')
print(f'   Metrica    : {modelo_knn.metric}')
print(f'   Algoritmo  : {modelo_knn.algorithm}')
print(f'   N vizinhos : {modelo_knn.n_neighbors}')
print(f'   Shape      : {matriz_esparsa.shape[0]:,} itens x {matriz_esparsa.shape[1]:,} usuarios')

---
## 8. Funcao de Recomendacao e Inferencia

A funcao abaixo encapsula toda a logica de recomendacao e constitui a interface de inferencia do modelo. Dado um espetaculo de referencia (`item_id`), ela retorna os `n` espetaculos mais similares, ordenados pela similaridade cosseno decrescente, com os nomes dos espetaculos recuperados dos metadados.

**Fluxo interno:**
1. Valida se o `item_id` existe no catalogo
2. Recupera o vetor do espetaculo na matriz esparsa
3. Consulta o KNN pelos `n+1` vizinhos (o +1 e necessario pois o proprio item aparece como vizinho de distancia zero)
4. Converte distancias em scores de similaridade: `sim = 1 - dist`
5. Faz join com metadados para exibir o nome do espetaculo
6. Retorna DataFrame ordenado por similaridade decrescente

In [ ]:
def recomendar_espetaculos(item_id, n_recomendacoes=10, verbose=True):
    """
    Recomenda espetaculos similares a um espetaculo-alvo usando KNN Item-Based.

    Esta funcao utiliza o modelo treinado (modelo_knn) e a matriz esparsa
    (matriz_esparsa) que devem estar carregados no ambiente. Para uso com
    o modelo exportado, carregue os artefatos com joblib antes de chamar
    esta funcao (ver Secao 11).

    Parametros
    ----------
    item_id : int
        ID numerico do espetaculo de referencia (indice codificado na Sprint 1).
    n_recomendacoes : int
        Numero de recomendacoes a retornar. Padrao: 10.
    verbose : bool
        Se True, exibe cabecalho com o espetaculo de referencia. Padrao: True.

    Retorna
    -------
    pd.DataFrame
        Colunas: item_id_recomendado | nome_evento | distancia_cosseno | similaridade
        Index  : rank (1 = mais similar)

    Raises
    ------
    ValueError
        Se o item_id nao existir no catalogo indexado.
    """

    if item_id not in item_to_idx:
        raise ValueError(
            f"item_id '{item_id}' nao encontrado no catalogo.\n"
            f"Utilize um dos {len(item_ids):,} itens disponiveis."
        )

    idx_alvo   = item_to_idx[item_id]
    vetor_alvo = matriz_esparsa[idx_alvo]     # vetor linha CSR

    # Busca n+1 vizinhos porque o próprio item aparece com distância zero.
    distancias, indices = modelo_knn.kneighbors(
        vetor_alvo,
        n_neighbors=n_recomendacoes + 1
    )

    distancias = distancias.flatten()
    indices    = indices.flatten()

    mascara    = indices != idx_alvo
    distancias = distancias[mascara][:n_recomendacoes]
    indices    = indices[mascara][:n_recomendacoes]

    similaridades = 1 - distancias

    recomendacoes = pd.DataFrame({
        'item_id_recomendado' : [idx_to_item[i] for i in indices],
        'distancia_cosseno'   : distancias,
        'similaridade'        : similaridades
    })

    recomendacoes = recomendacoes.merge(
        item_meta[['item_id', 'nome_evento', 'categoria']],
        left_on='item_id_recomendado',
        right_on='item_id',
        how='left'
    ).drop(columns=['item_id'])

    recomendacoes = recomendacoes.sort_values('similaridade', ascending=False).reset_index(drop=True)
    recomendacoes.index      = range(1, len(recomendacoes) + 1)
    recomendacoes.index.name = 'rank'

    if verbose:
        nome_ref = item_meta.loc[item_meta['item_id'] == item_id, 'nome_evento']
        nome_str = nome_ref.values[0] if len(nome_ref) > 0 else f'ID {item_id}'
        print(f'=== Recomendacoes para: {nome_str} (ID {item_id}) ===')
        print(f'   Top {n_recomendacoes} espetaculos mais similares:')

    return recomendacoes

print('Funcao de recomendacao definida com sucesso')


In [ ]:
# Exemplo 1: espet?culo mais avaliado do cat?logo
item_exemplo_1 = df['item_id'].value_counts().index[0]
rec1 = recomendar_espetaculos(item_exemplo_1, n_recomendacoes=10)
display(rec1)


In [ ]:
# Exemplo 2: segundo espet?culo mais avaliado
item_exemplo_2 = df['item_id'].value_counts().index[1]
rec2 = recomendar_espetaculos(item_exemplo_2, n_recomendacoes=10)
display(rec2)


In [ ]:
# Exemplo 3: terceiro espet?culo mais avaliado
item_exemplo_3 = df['item_id'].value_counts().index[2]
rec3 = recomendar_espetaculos(item_exemplo_3, n_recomendacoes=10)
display(rec3)


### Leitura dos Exemplos de Infer?ncia

Os exemplos de recomenda??o demonstram que a fun??o de infer?ncia retorna listas coerentes a partir de espet?culos de refer?ncia, com itens ordenados por similaridade cosseno e enriquecidos com nome e categoria. Como o modelo ? puramente colaborativo, a proximidade entre espet?culos reflete padr?es de cointera??o entre usu?rios, e n?o semelhan?a textual ou curadoria manual de conte?do.

Esse comportamento ? adequado para um primeiro modelo interpret?vel do portal, mas tamb?m evidencia uma limita??o: espet?culos novos ou pouco avaliados tendem a receber recomenda??es menos confi?veis at? acumularem intera??es suficientes.


In [ ]:
# Mapa de similaridade entre os Top 15 espetáculos mais avaliados
TOP_N_VISUAL = 15

top_itens_visual = df['item_id'].value_counts().head(TOP_N_VISUAL).index.tolist()
indices_top      = [item_to_idx[i] for i in top_itens_visual]
submatriz        = matriz_esparsa[indices_top]

# Calcular matriz de similaridade cosseno entre os top itens
sim_matrix = cosine_similarity(submatriz)

# Recuperar nomes dos espetaculos para o eixo
nomes_top = []
for iid in top_itens_visual:
    nome = item_meta.loc[item_meta['item_id'] == iid, 'nome_evento']
    nomes_top.append(nome.values[0][:20] if len(nome) > 0 else f'ID {iid}')

sim_df = pd.DataFrame(sim_matrix, index=nomes_top, columns=nomes_top)

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(sim_df, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Similaridade Cosseno'},
            annot_kws={'size': 7})
ax.set_title(
    f'Mapa de Similaridade — Top {TOP_N_VISUAL} Espetaculos (Portal Aventura)',
    fontsize=12, fontweight='bold', pad=15
)
ax.set_xlabel('Espetaculo', fontsize=10)
ax.set_ylabel('Espetaculo', fontsize=10)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('mapa_similaridade_knn.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 9. Avalia??o do Modelo: Erro, Cobertura e Ranking

### Como avaliar quantitativamente o KNN Item-Based?

O KNN nao prevê ratings diretamente como um modelo de regressao. Para avaliacao quantitativa, adotamos a seguinte estrategia:

1. Dividimos o dataset em treino (80%) e teste (20%)
2. Treinamos um KNN apenas com os dados de treino — sem acesso ao conjunto de teste
3. Para cada par `(user_id, item_id)` no teste, estimamos o rating usando a media ponderada pela similaridade dos `k` itens vizinhos que o usuario avaliou:

$$\hat{r}_{u,i} = \frac{\sum_{j \in N(i)} \text{sim}(i,j) \cdot r_{u,j}}{\sum_{j \in N(i)} |\text{sim}(i,j)|}$$

Onde $N(i)$ e o conjunto dos $k$ itens mais similares ao item $i$ que o usuario $u$ ja avaliou.

4. Calculamos **RMSE** e **MAE** entre os ratings reais e previstos:

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(\hat{r}_i - r_i)^2} \qquad MAE = \frac{1}{n}\sum_{i=1}^{n}|\hat{r}_i - r_i|$$

**RMSE** penaliza erros grandes de forma quadratica, sendo sensivel a outliers de predicao. **MAE** representa o erro absoluto medio em unidades da escala original (1 a 5), sendo de interpretacao mais direta. Ambas as metricas sao adequadas para avaliar modelos de recomendacao baseados em rating.

Al?m dessas m?tricas de erro, reportamos a **cobertura**: percentual de pares no conjunto de teste para os quais foi poss?vel gerar uma predi??o. Pares sem vizinhos com avalia??es em comum n?o geram predi??o e s?o exclu?dos do c?lculo de RMSE/MAE.

Tamb?m inclu?mos uma m?trica orientada a ranking, o **Hit Rate@10**. Para cada usu?rio avaliado, usamos um item observado no treino como refer?ncia e verificamos se algum item retido no teste aparece entre as 10 recomenda??es. Essa m?trica aproxima melhor o uso real do sistema, pois avalia se a lista recomendada recupera itens que o usu?rio de fato consumiu.


In [ ]:
# Avaliação holdout do KNN
print('=== Avaliacao do Modelo KNN ===')
print('   Dividindo dados em treino (80%) e teste (20%)...')

df_treino, df_teste = train_test_split(
    df, test_size=0.2, random_state=42
)

print(f'   Treino : {len(df_treino):,} interacoes')
print(f'   Teste  : {len(df_teste):,} interacoes')

In [ ]:
# A matriz de avaliação usa apenas treino para evitar vazamento de dados.
matriz_treino = df_treino.pivot_table(
    index='item_id',
    columns='user_id',
    values='rating',
    aggfunc='mean'
).fillna(0)

matriz_treino_esparsa = csr_matrix(matriz_treino.values)

# Mapeamentos especificos para a matriz de treino
item_ids_treino = list(matriz_treino.index)
user_ids_treino = list(matriz_treino.columns)
item_to_idx_tr  = {item: idx for idx, item in enumerate(item_ids_treino)}
idx_to_item_tr  = {idx: item for item, idx in item_to_idx_tr.items()}
user_to_idx_tr  = {user: idx for idx, user in enumerate(user_ids_treino)}

K_AVALIACAO = 10   # numero de vizinhos para predicao

knn_avaliacao = NearestNeighbors(
    n_neighbors=min(K_AVALIACAO + 1, len(item_ids_treino)),
    metric='cosine',
    algorithm='brute',
    n_jobs=-1
)
knn_avaliacao.fit(matriz_treino_esparsa)

print('KNN treinado com dados de treino para avaliacao')


In [ ]:
def prever_rating(user_id, item_id, k=K_AVALIACAO):
    """
    Estima o rating de um usuario para um espetaculo usando
    a media ponderada pela similaridade cosseno dos k vizinhos.
    Retorna NaN se nao houver informacao suficiente.
    """
    if item_id not in item_to_idx_tr or user_id not in user_to_idx_tr:
        return np.nan

    idx_item = item_to_idx_tr[item_id]
    idx_user = user_to_idx_tr[user_id]

    dists, idxs = knn_avaliacao.kneighbors(
        matriz_treino_esparsa[idx_item], n_neighbors=k + 1
    )
    dists = dists.flatten()
    idxs  = idxs.flatten()

    mask  = idxs != idx_item
    dists = dists[mask][:k]
    idxs  = idxs[mask][:k]

    sims = 1 - dists

    ratings_viz = np.array([
        matriz_treino_esparsa[i, idx_user] for i in idxs
    ])

    # Só vizinhos já avaliados pelo usuário contribuem para a predição.
    avaliados = ratings_viz > 0
    if not avaliados.any():
        return np.nan

    soma_similaridades = sims[avaliados].sum()
    if soma_similaridades == 0:
        return np.nan

    return np.dot(sims[avaliados], ratings_viz[avaliados]) / soma_similaridades


In [ ]:
# Amostra de até 500 registros para manter a execução leve em ambiente acadêmico.
N_AMOSTRA        = min(500, len(df_teste))
df_teste_amostra = df_teste.sample(N_AMOSTRA, random_state=42)

print(f'Calculando predicoes para {N_AMOSTRA:,} registros do conjunto de teste...')

y_real     = []
y_previsto = []

for _, row in df_teste_amostra.iterrows():
    pred = prever_rating(row['user_id'], row['item_id'])
    if not np.isnan(pred):
        y_real.append(row['rating'])
        y_previsto.append(pred)

rmse      = np.sqrt(mean_squared_error(y_real, y_previsto))
mae       = mean_absolute_error(y_real, y_previsto)
cobertura = len(y_real) / N_AMOSTRA * 100

print()
print('=== Resultados — KNN Item-Based (Portal Aventura) ===')
print(f'   RMSE       : {rmse:.4f}')
print(f'   MAE        : {mae:.4f}')
print(f'   Cobertura  : {cobertura:.1f}% ({len(y_real)}/{N_AMOSTRA} predicoes validas)')
print()
print('   Interpretacao das metricas:')
print(f'     RMSE = {rmse:.4f} -> erro quadratico medio entre rating real e previsto (penaliza erros grandes)')
print(f'     MAE  = {mae:.4f} -> em media, a predicao erra {mae:.2f} ponto(s) na escala de 1 a 5')
print(f'     Cobertura -> percentual de pares (usuario, item) com vizinhos suficientes para predicao')

# Salvar metricas para comparacao futura com SVD
metricas_knn = {
    'modelo'    : 'KNN Item-Based',
    'RMSE'      : round(rmse, 4),
    'MAE'       : round(mae, 4),
    'Cobertura' : round(cobertura, 1)
}


### Avalia??o de Ranking com Hit Rate@10

Como o uso final do modelo ? recomendar uma lista de espet?culos similares, complementamos RMSE, MAE e cobertura com uma m?trica de ranking. O Hit Rate@10 mede a propor??o de usu?rios avali?veis para os quais ao menos um item do teste aparece entre as 10 recomenda??es geradas a partir de um item observado no treino.


In [ ]:
K_RANKING = 10

def recomendar_itens_avaliacao(item_id, n_recomendacoes=K_RANKING):
    if item_id not in item_to_idx_tr:
        return []

    idx_alvo = item_to_idx_tr[item_id]
    n_vizinhos = min(n_recomendacoes + 1, len(item_ids_treino))
    distancias, indices = knn_avaliacao.kneighbors(
        matriz_treino_esparsa[idx_alvo],
        n_neighbors=n_vizinhos
    )

    indices = indices.flatten()
    indices = [idx for idx in indices if idx != idx_alvo][:n_recomendacoes]
    return [idx_to_item_tr[idx] for idx in indices]

hits = 0
casos_ranking = 0

for user_id, grupo_teste in df_teste_amostra.groupby('user_id'):
    historico_treino = df_treino[df_treino['user_id'] == user_id]
    historico_treino = historico_treino[historico_treino['item_id'].isin(item_to_idx_tr)]

    if historico_treino.empty:
        continue

    item_referencia = historico_treino.sort_values('rating', ascending=False).iloc[0]['item_id']
    recomendados = set(recomendar_itens_avaliacao(item_referencia, K_RANKING))
    itens_teste = set(grupo_teste['item_id'])

    if not recomendados or not itens_teste:
        continue

    casos_ranking += 1
    hits += int(len(recomendados.intersection(itens_teste)) > 0)

hit_rate_10 = hits / casos_ranking * 100 if casos_ranking else 0

print('=== Avalia??o de Ranking ? KNN Item-Based ===')
print(f'   Hit Rate@10 : {hit_rate_10:.1f}% ({hits}/{casos_ranking} usu?rios com acerto)')

metricas_knn['Hit Rate@10'] = round(hit_rate_10, 1)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Avaliacao do Modelo KNN Item-Based — Portal Aventura',
             fontsize=13, fontweight='bold')

# Scatter: real vs previsto
axes[0].scatter(y_real, y_previsto, alpha=0.35, color='steelblue', s=18)
lims = [min(min(y_real), min(y_previsto)), max(max(y_real), max(y_previsto))]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Predicao Perfeita')
axes[0].set_xlabel('Rating Real')
axes[0].set_ylabel('Rating Previsto')
axes[0].set_title('Rating Real vs Rating Previsto', fontweight='bold')
axes[0].legend()
axes[0].text(0.05, 0.90,
             f'RMSE: {rmse:.3f}\nMAE : {mae:.3f}\nCob.: {cobertura:.1f}%',
             transform=axes[0].transAxes,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
             fontsize=10)

# Histograma de erros
erros = np.array(y_previsto) - np.array(y_real)
axes[1].hist(erros, bins=30, color='salmon', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Erro Zero')
axes[1].axvline(erros.mean(), color='navy', linestyle=':', linewidth=1.5,
                label=f'Media: {erros.mean():.2f}')
axes[1].set_xlabel('Erro (Previsto - Real)')
axes[1].set_ylabel('Frequencia')
axes[1].set_title('Distribuicao dos Erros de Predicao', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('avaliacao_knn.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 10. Preparacao para Comparacao com SVD

Para facilitar a comparacao futura com SVD Classico e Funk SVD, estruturamos uma tabela comparativa padronizada que sera preenchida com os resultados dos modelos subsequentes.

### Diferencas conceituais: KNN vs SVD

| Aspecto | KNN Item-Based | SVD Classico | Funk SVD |
|---|---|---|---|
| Tipo | Memoria (lazy learning) | Modelo (fatorizacao) | Modelo (fatorizacao) |
| Treinamento | Indexacao da matriz | Decomposicao SVD completa | SGD iterativo |
| Predicao | Busca por vizinhos | Produto de vetores latentes | Produto de vetores latentes |
| Esparsidade | Sensivel — cobertura limitada | Lida bem — fatoriza a matriz densa | Lida bem — opera diretamente nos pares observados |
| Escalabilidade | O(n²) na predicao | O(k) na predicao apos treino | O(k) na predicao apos treino |
| Interpretabilidade | Alta — vizinhos sao espetaculos reais | Baixa — fatores latentes sem significado direto | Baixa — fatores latentes sem significado direto |
| Hiperparametros | k, metrica | n_fatores | n_fatores, learning_rate, regularizacao |

O SVD resolve a principal limitacao do KNN neste contexto: a esparsidade de 98,62% da matriz reduz a cobertura das predicoes do metodo baseado em memoria. Modelos de fatorizacao matricial operam sobre os padroes latentes dos dados e podem gerar predicoes mesmo em regioes esparsas.

In [ ]:
# Tabela comparativa: KNN preenchido; SVD fica para etapa posterior
resultados_comparativos = [
    {
        'Modelo'          : 'KNN Item-Based',
        'Tipo'            : 'Memoria',
        'RMSE'            : round(rmse, 4),
        'MAE'             : round(mae, 4),
        'Cobertura (%)'   : round(cobertura, 1),
        'Hit Rate@10 (%)' : round(hit_rate_10, 1),
        'Hiperparametros' : f'k={N_VIZINHOS}, metrica=cosseno'
    },
    {
        'Modelo'          : 'SVD Classico',
        'Tipo'            : 'Fatorizacao Matricial',
        'RMSE'            : '-',
        'MAE'             : '-',
        'Cobertura (%)'   : '-',
        'Hit Rate@10 (%)' : '-',
        'Hiperparametros' : 'n_fatores=?'
    },
    {
        'Modelo'          : 'Funk SVD',
        'Tipo'            : 'Fatorizacao Matricial',
        'RMSE'            : '-',
        'MAE'             : '-',
        'Cobertura (%)'   : '-',
        'Hit Rate@10 (%)' : '-',
        'Hiperparametros' : 'n_fatores=?, lr=?, reg=?'
    }
]

df_comparativo = pd.DataFrame(resultados_comparativos)

display(df_comparativo.set_index('Modelo'))

# Salvar para uso em compara??es futuras
df_comparativo.to_csv('comparativo_modelos.csv', index=False)


---
## 11. Exportacao do Modelo

Serializamos todos os artefatos necessarios para reproducao e futura integracao com o Portal Aventura usando `joblib`, preferido ao `pickle` para objetos NumPy e Scipy por sua eficiencia com arrays de grande dimensao.

Como este notebook fica em `src/aad`, os artefatos finais do KNN sao gravados em `src/aad/modelos`. Os arquivos `eda_modelagem.png`, `mapa_similaridade_knn.png`, `avaliacao_knn.png` e `comparativo_modelos.csv` permanecem em `src/aad` como saidas auxiliares locais da modelagem. Ja os arquivos em `src/dataset` pertencem ao notebook de tratamento e representam bases preparadas, nao resultados do modelo KNN.

| Arquivo | Conteudo | Uso |
|---|---|---|
| `modelo_knn.pkl` | Objeto NearestNeighbors treinado | Geracao de recomendacoes |
| `matriz_item_usuario.pkl` | Pivot Table original | Referencia e diagnostico |
| `matriz_esparsa.pkl` | Matriz CSR usada no treino | Input para kneighbors() |
| `mapeamentos_itens.pkl` | Dicionarios indice para item_id | Decodificacao dos resultados |


In [ ]:
# Exportação dos artefatos necessários para inferência
os.makedirs('modelos', exist_ok=True)

joblib.dump(modelo_knn, 'modelos/modelo_knn.pkl')
print('Modelo KNN salvo          -> modelos/modelo_knn.pkl')

joblib.dump(matriz_item_usuario, 'modelos/matriz_item_usuario.pkl')
print('Matriz pivot salva        -> modelos/matriz_item_usuario.pkl')

joblib.dump(matriz_esparsa, 'modelos/matriz_esparsa.pkl')
print('Matriz esparsa salva      -> modelos/matriz_esparsa.pkl')

mapeamentos = {
    'item_ids'    : item_ids,
    'item_to_idx' : item_to_idx,
    'idx_to_item' : idx_to_item
}
joblib.dump(mapeamentos, 'modelos/mapeamentos_itens.pkl')
print('Mapeamentos salvos        -> modelos/mapeamentos_itens.pkl')

print('\n=== Artefatos Gerados ===')
for f in sorted(os.listdir('modelos')):
    tamanho = os.path.getsize(f'modelos/{f}') / 1024
    print(f'   {f:<40} {tamanho:>8.1f} KB')


In [ ]:
# Verificação de carregamento em outro contexto

print('Testando carregamento do modelo salvo...')

modelo_carregado = joblib.load('modelos/modelo_knn.pkl')
mapeamentos_load = joblib.load('modelos/mapeamentos_itens.pkl')
matriz_load      = joblib.load('modelos/matriz_esparsa.pkl')

# Teste funcional: buscar vizinhos do primeiro item do catalogo
item_teste  = mapeamentos_load['item_ids'][0]
idx_teste   = mapeamentos_load['item_to_idx'][item_teste]
dists, idxs = modelo_carregado.kneighbors(matriz_load[idx_teste], n_neighbors=3)

print(f'Modelo carregado e validado com sucesso')
print(f'   Item de teste : ID {item_teste}')
print(f'   Vizinhos      : {idxs.flatten().tolist()}')
print(f'   Distancias    : {[round(d, 4) for d in dists.flatten().tolist()]}')

---
## 12. Conclusao

### S?ntese da Entrega

O notebook implementa um modelo de filtragem colaborativa baseada em itens com KNN para o Portal Aventura. A solu??o usa o dataset balanceado da Sprint 1, constr?i uma matriz item-usu?rio, treina o `NearestNeighbors` com dist?ncia cosseno, disponibiliza uma fun??o de infer?ncia funcional e exporta os artefatos necess?rios para reuso do modelo.

A checagem explorat?ria desta etapa foi mantida de forma objetiva porque a explora??o detalhada e o tratamento dos dados j? foram realizados em `tratamento-dados.ipynb`. Aqui, a an?lise se concentra nos aspectos que impactam diretamente o KNN: volume de usu?rios e itens, distribui??o dos ratings, densidade da matriz e quantidade de intera??es por usu?rio e espet?culo.

### Leitura dos Resultados

A matriz final apresentou 40.848 intera??es, 8.648 usu?rios, 343 espet?culos e esparsidade de 98,6229%. Esse n?vel de esparsidade ? esperado em sistemas de recomenda??o, mas ? o principal fator limitante do KNN: quando o usu?rio n?o possui avalia??es em comum com itens vizinhos, o modelo n?o consegue estimar o rating.

Na avalia??o com split 80/20 e amostra de 500 registros do conjunto de teste, o modelo obteve RMSE de 1,6826, MAE de 1,3013 e cobertura de 55,4% (277 predi??es v?lidas em 500 casos avaliados). O MAE indica que, quando h? predi??o, o erro m?dio ? de aproximadamente 1,30 ponto na escala de 1 a 5. A cobertura mostra que quase metade dos pares avaliados n?o teve vizinhan?a suficiente para gerar estimativa, refor?ando a esparsidade como a maior fragilidade operacional do KNN.

Como m?trica de ranking, o modelo alcan?ou Hit Rate@10 de 26,2% (123 acertos em 469 usu?rios avali?veis). Isso significa que, em cerca de um quarto dos casos avali?veis, ao menos um item retido no teste apareceu entre as 10 recomenda??es geradas a partir do hist?rico de treino do usu?rio. O resultado ? compat?vel com um baseline colaborativo interpret?vel, mas ainda indica espa?o para modelos mais robustos ? esparsidade.

### Pontos Fortes e Limita??es

O principal ponto forte da abordagem ? a coer?ncia entre o problema e o modelo: recomenda??es item-based s?o interpret?veis e adequadas ao contexto do portal, pois permitem sugerir espet?culos similares a partir de um item acessado, mesmo sem hist?rico longo do usu?rio. A fun??o de infer?ncia tamb?m est? funcional, retorna itens ordenados por similaridade e enriquece os resultados com nome e categoria dos espet?culos.

A principal limita??o est? na depend?ncia de coavalia??es entre usu?rios e itens. O KNN ? sens?vel ? esparsidade e, por isso, apresenta cobertura limitada. O Hit Rate@10 fortalece a avalia??o por aproximar o cen?rio de uso real, mas ainda ? uma m?trica inicial de ranking. Para uma avalia??o mais completa do ponto de vista do usu?rio final, m?tricas como Precision@K, Recall@K e NDCG@K devem ser consideradas em uma pr?xima vers?o.

---

### Dataset e Resultados

| Metrica | Valor |
|---|---|
| Registros brutos originais | 343.758 |
| Registros apos tratamento Sprint 1 | 40.848 |
| Usuarios no dataset final | 8.648 |
| Espetaculos no dataset final | 343 |
| Esparsidade da matriz | 98,6229% |
| Vizinhos utilizados (k) | 20 |
| Metrica de similaridade | Cosseno |
| Split treino/teste | 80% / 20% |
| Amostra de avalia??o | 500 registros do teste |
| RMSE (teste) | 1,6826 |
| MAE (teste) | 1,3013 |
| Cobertura (%) | 55,4% |
| Hit Rate@10 | 26,2% |

---

### Proximos Passos

1. **Avaliar o conjunto de teste completo**: remover a amostragem de 500 registros quando houver tempo computacional suficiente;
2. **Ampliar m?tricas de ranking**: complementar Hit Rate@10 com Precision@K, Recall@K e NDCG@K;
3. **Comparar com SVD e Funk SVD**: verificar se modelos de fatoriza??o aumentam a cobertura em uma matriz muito esparsa;
4. **Integrar metadados dos espet?culos**: combinar filtragem colaborativa com categoria, teatro e outras informa??es de conte?do para reduzir problemas de cold start;
5. **Integrar ao portal**: consumir os artefatos exportados em `src/aad/modelos` em uma API ou servi?o de recomenda??o.
